# Variant Effect Prediction with Coding Agents
### Upper Bound Workshop — Agentic Research Workflows

Welcome. Over the next ~3 hours we walk through a single research workflow end to end,
using protein variant effect prediction (VEP) as the worked example:

- **S1 — Setup & Orientation.** Load 500 ClinVar missense variants + WT sequences. Quick EDA.
- **S2 — Prototype.** Score one pathogenic + one benign variant live with ESM-1b. Two
  embeddings, four numbers, one plot.
- **S3 — Validate.** Same prototype, all 500 variants. AUROC + CIs + four diagnostic plots.
- **S4 — Hand off.** Three concrete agent prompts: write a paper around the prototype,
  **sweep it on wandb (60 runs from one prompt → paper-worthy figure)**, or rewrite it to
  use a hosted encoder for no-GPU users.

Each section has a 🤖 **Agent prompt** block — paste it into Claude Code (or read it as
narrative). The lesson is the **prototype → validate → hand off** pattern; everything we
do at the protein level here generalizes one step up the dogma. Vocabulary lives in the
slides; this notebook is what you actually run.

---


## Glossary

Five terms you'll see throughout. Read once; come back if anything below stops making sense.

- **VEP** — predicting whether a DNA mutation breaks the protein it codes for.
- **ESM-1b** — a model trained on hundreds of millions of natural protein sequences. We use it to score how unusual a mutation looks.
- **LLR** — a number that answers "how surprised is ESM-1b by this mutation, given the surrounding sequence?" Higher means more surprised, which usually means more likely to break the protein.
- **Harness** — the opinionated codebase that lets you or an AI agent drive an experiment without re-figuring out the plumbing each time.
- **manylatents / manylatents-omics** — the libraries this repo uses. The first is general-purpose; the second adds biological encoders and scoring. Both are pip-installable.

## S1. Setup & Orientation

**The workflow is the lesson.** You'll prototype a small protein-VEP pipeline in this notebook,
see whether the signal holds, and then hand the prototype off to a coding agent to scale and
extend. Vocabulary — what an *agent* is, what a *harness* is, where the human stays in the
loop — lives in the slide deck. This notebook is what you actually run.


> 🟢 **Runtime: T4 GPU (set as default).** If you opened this notebook from the GitHub link, Colab should already have a T4 GPU runtime selected for you. Confirm via `Runtime → Change runtime type` and look for **T4 GPU**.
>
> ESM-1b is a 650M-param protein language model. On a T4 the demo-pair encode runs in ~2 s and the full 500-variant scoring in ~4 min. On the default CPU runtime the same work takes 10–30× longer. CPU still works — figure cells reading the cached scores don't need GPU at all.


In [ ]:
# Pin the environment so the notebook runs identically on Colab and locally.
#
# Encoder backend is fair-esm — Facebook's official ESM library, the one
# Brandes 2023 used. Single install, no transformers dependency chain, no
# tokenizers version pin, no numpy.strings ABI fight with Colab's stack.
# Bit-identical LLR to the HuggingFace transformers path (verified).
#
# LOCAL: assumes `uv sync` from the repo root has run. The uv venv has
#        pinned deps + doesn't ship pip; this cell is a no-op.
# COLAB: installs fair-esm. fair-esm only depends on torch (already
#        preinstalled on Colab), so this is fast and clean.
import sys
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    %pip install -q fair-esm
else:
    print('  Local env (uv-managed) — s1-install is a no-op; deps pinned in pyproject.toml + uv.lock.')

# Diagnostic — confirm fair-esm is importable.
import esm
esm_ver = getattr(esm, '__version__', '(unknown)')
print(f'  fair-esm {esm_ver}  •  ready ✓')

In [ ]:
# Pull the workshop assets: utility module, validation set, WT protein FASTA.
# All three live in the workshop repo; together they're under 400 KB.
import os
from pathlib import Path

REPO_RAW = "https://raw.githubusercontent.com/latent-reasoning-works/lrw-vep-ub2026/main/experiments/notebooks"
ASSETS = {
    "vep_utils.py": f"{REPO_RAW}/vep_utils.py",
    "data/workshop_set.tsv": f"{REPO_RAW}/data/workshop_set.tsv",
    "data/workshop_set_proteins.fasta": f"{REPO_RAW}/data/workshop_set_proteins.fasta",
    "data/demo_pair.tsv": f"{REPO_RAW}/data/demo_pair.tsv",
}
Path("data").mkdir(exist_ok=True)
for name, url in ASSETS.items():
    if not Path(name).exists():
        !wget -q -O {name} {url}
        print(f"  downloaded {name}")
    else:
        print(f"  cached    {name}")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from vep_utils import load_fasta

# Variant table: variant_id, gene, protein_pos (0-indexed), aa_ref, aa_alt,
#                clinical_significance, label (1=pathogenic, 0=benign),
#                uniprot_id, isoform_validated  (last two are audit metadata)
df = pd.read_csv("data/workshop_set.tsv", sep="\t")

# WT protein sequences keyed by variant_id (see vep_utils.load_fasta for the parser).
wt_seqs = load_fasta("data/workshop_set_proteins.fasta")
assert set(df["variant_id"]) <= set(wt_seqs), "every variant needs a WT sequence"

print(f"Variants : {len(df)}  ({(df['label']==1).sum()} pathogenic / {(df['label']==0).sum()} benign)")
print(f"Genes    : {df['gene'].nunique()} unique")
print(f"Sequences: {len(wt_seqs)} WT proteins, median {int(np.median([len(s) for s in wt_seqs.values()]))} aa")


In [ ]:
# Quick orientation: class balance, sequence length distribution, gene diversity.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.countplot(x="label", data=df, ax=axes[0])
axes[0].set_xticklabels(["Benign (0)", "Pathogenic (1)"])
axes[0].set_title("Class balance")

seq_lens = np.array([len(wt_seqs[v]) for v in df["variant_id"]])
axes[1].hist(seq_lens, bins=40, color="#3a7ca5")
axes[1].axvline(1024, color="crimson", linestyle="--", linewidth=1, label="ESM-1b max")
axes[1].set_xlabel("Protein length (aa)")
axes[1].set_title(f"Sequence lengths (median {int(np.median(seq_lens))} aa)")
axes[1].legend()

gene_counts = df["gene"].value_counts()
axes[2].barh(gene_counts.head(10).index[::-1], gene_counts.head(10).values[::-1], color="#3a7ca5")
axes[2].set_xlabel("Variant count")
axes[2].set_title(f"Top 10 of {df['gene'].nunique()} genes")
plt.tight_layout()
plt.show()

print(f"\nVariants exceeding ESM-1b's 1024-token window: {(seq_lens > 1024).sum()} of {len(seq_lens)}")


> 🤖 **Agent prompt** (paste into Claude Code, or read as narrative)
>
> *"What's in the validation set we're about to score? I want to know what would
> surprise me — class balance, anything skewed, anything that won't fit the encoder,
> anything I'd want to flag in a footnote. Surface the surprises; skip the obvious."*

This is the smallest useful agent task: **data sanity**. ~2 min of model time replaces
30 min of squinting at columns. Use the same pattern any time you inherit a dataset.


---

## S2. Prototype: One Pathogenic, One Benign

We'll load **ESM-1b** — Meta's general-purpose protein language model — and score one
pathogenic and one benign variant end-to-end. Two encodings each, four numbers, one plot.
That's the prototype. ESM-1b is fully open (no auth), fits on a Colab T4, and (per Brandes
et al.) outperforms the VEP-specific ESM-1v.

**Heads up on indexing**: our CSV stores `protein_pos` as 0-indexed (matches the encoding
pipeline). `vep_utils.parse_mutation` expects HGVS-style 1-indexed strings (e.g., `D67A`).
We convert with `+1` at the boundary.


In [ ]:
from vep_utils import ESM1bEncoder, encode_variant, parse_mutation, validate_mutation

encoder = ESM1bEncoder()  # auto-detects GPU


In [ ]:
if 'df' not in globals():
    raise RuntimeError("Run the S1 cells first — `df` and `wt_seqs` need to exist.")

# S2 uses a fixed canonical pair (BRCA1 L1854P pathogenic + P1859R benign) so the
# prototype is deterministic regardless of which 500-variant draw landed in
# workshop_set.tsv. Validation at scale still uses `df` in S3.
demo = pd.read_csv("data/demo_pair.tsv", sep="\t")
row_p = demo[demo['label'] == 1].iloc[0]
row_b = demo[demo['label'] == 0].iloc[0]

def hgvs(row):
    # CSV is 0-indexed; HGVS is 1-indexed
    return f"{row['aa_ref']}{int(row['protein_pos']) + 1}{row['aa_alt']}"

print(f"Pathogenic: {row_p['gene']} {hgvs(row_p)}  ({row_p['variant_id']}, {row_p['clinical_significance']})")
print(f"Benign    : {row_b['gene']} {hgvs(row_b)}  ({row_b['variant_id']}, {row_b['clinical_significance']})")


> 🤖 **Agent prompt** (paste into Claude Code, or read as narrative)
>
> *"Encode the demo pair and tell me whether ESM-1b separates the pathogenic from
> the benign. Show me whatever convinces me. If it doesn't separate them, the rest
> of the demo doesn't hold — flag it."*


In [ ]:
# Encode both variants. Each call runs ESM-1b twice (WT + MUT).
# Mirror S3's truncate-around-mutation pattern so long proteins (>MAX_LEN
# residues) are center-windowed and the HGVS position is re-indexed to the
# truncated window.
from vep_utils import truncate_around_mutation, compute_delta_norm

results = {}
for label, row in [('P', row_p), ('B', row_b)]:
    seq = wt_seqs[row['variant_id']]
    pos1 = int(row['protein_pos']) + 1  # 0-indexed -> 1-indexed for HGVS
    seq_t, pos_t = truncate_around_mutation(seq, pos1, window=encoder.MAX_LEN)
    mut_str = f"{row['aa_ref']}{pos_t}{row['aa_alt']}"
    # Sanity-check the mutation lands on the right WT residue
    validate_mutation(seq_t, parse_mutation(mut_str))
    results[label] = encode_variant(encoder, seq_t, mut_str)
    d = results[label]['mut_embedding'] - results[label]['wt_embedding']
    norm = compute_delta_norm(results[label]['wt_embedding'], results[label]['mut_embedding'])
    print(f"  {label}  delta L2 norm = {norm:.3f}   nonzero dims = {(d != 0).sum()} / {len(d)}")


In [ ]:
# Two panels: (1) delta norm bar chart; (2) top-10 dimensions per variant.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

deltas = {k: results[k]['mut_embedding'] - results[k]['wt_embedding'] for k in results}
norms = {k: float(np.linalg.norm(d)) for k, d in deltas.items()}

ax = axes[0]
ax.bar(['Benign', 'Pathogenic'], [norms['B'], norms['P']],
       color=['#4a8fb8', '#c44e52'])
ax.set_ylabel('Delta L2 norm')
ax.set_title('Embedding perturbation: P vs B')
for i, k in enumerate(['B', 'P']):
    ax.text(i, norms[k], f"{norms[k]:.2f}", ha='center', va='bottom')

# Top-10 changing dimensions for the pathogenic variant; show the same indices for benign for comparison.
ax = axes[1]
top10_p = np.argsort(np.abs(deltas['P']))[-10:][::-1]
x = np.arange(10)
w = 0.4
ax.barh(x - w/2, deltas['B'][top10_p], height=w, color='#4a8fb8', label='Benign')
ax.barh(x + w/2, deltas['P'][top10_p], height=w, color='#c44e52', label='Pathogenic')
ax.set_yticks(x); ax.set_yticklabels([f'dim {i}' for i in top10_p])
ax.invert_yaxis()
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('Delta value (mut − wt)')
ax.set_title("Pathogenic's top-10 changed dimensions")
ax.legend()
plt.tight_layout()
plt.show()


**What just happened**: One run of ESM-1b on each WT/MUT pair — four forward passes total.
The pathogenic variant produced a larger embedding shift in this case, but **n=2 is an
anecdote, not a result**. To know if the gap holds, we need to do this at scale. That's S3.

**The lesson here**: this is exactly the kind of small, focused prototype you'd hand to a
coding agent: a self-contained function (`encode_variant`), a clear input contract (a
sequence and an HGVS-p string), and one clear output (two embeddings). Whatever the agent
does next — sweep, deploy, write up — starts from this primitive.


In [ ]:
# Try your own: pass any variant_id from df (or paste a custom sequence + mutation).
def score_one(variant_id: str | None = None,
              sequence: str | None = None,
              mutation: str | None = None):
    if variant_id is not None:
        row = df[df['variant_id'] == variant_id].iloc[0]
        sequence = wt_seqs[variant_id]
        mutation = f"{row['aa_ref']}{int(row['protein_pos']) + 1}{row['aa_alt']}"
        print(f"  variant_id={variant_id}  gene={row['gene']}  label={'P' if row['label']==1 else 'B'}")
    if sequence is None or mutation is None:
        raise ValueError("Pass either variant_id, or both sequence and mutation.")
    r = encode_variant(encoder, sequence, mutation)
    norm = float(np.linalg.norm(r['mut_embedding'] - r['wt_embedding']))
    print(f"  mutation={mutation}  delta L2 norm = {norm:.3f}")
    return r

# Example: score_one(variant_id=df.sample(1, random_state=0).iloc[0]['variant_id'])


---

## S3. Validate: Does the Signal Hold at Scale?

S2 was n=2 — a hint, not a result. Now we run the same prototype across all 500 variants
(250 P / 250 B) and ask: **does the gap between pathogenic and benign perturbations hold
at the protein level with ESM-1b?**

Two scores per variant, computed live:
- **Delta L2 norm** — *how far did the embedding move?*
- **LLR** — *how surprised is the model by the mutation at that position?*

On a single H100 this is a couple of minutes; on a Colab T4, ~15 min. The cell caches its
output to `data/s3_scores.npz` so re-runs are instant.


> 🤖 **Agent prompt**
>
> *"Run S2's pattern across all 500 variants — actually run it end-to-end,
> not a cache read. Write the result to `data/s3_scores.npz` as output so the
> figure pass can reload it. Tell me whether pathogenic-vs-benign separation
> actually generalizes — and where it breaks (which gene, which length
> regime, which mutation class)."*


In [ ]:
from pathlib import Path
from vep_utils import (
    ESM1bEncoder, encode_variant, parse_mutation, validate_mutation,
    truncate_around_mutation, compute_delta_norm, compute_llr,
)

CACHE = Path("data/s3_scores.npz")
if CACHE.exists():
    z = np.load(CACHE, allow_pickle=True)
    scores_df = pd.DataFrame({
        'variant_id': z['variant_id'], 'gene': z['gene'], 'label': z['label'],
        'delta_norm': z['delta_norm'], 'llr': z['llr'], 'seq_len': z['seq_len'],
    })
    print(f"Loaded cached scores: {len(scores_df)} variants")
else:
    if 'encoder' not in globals():
        encoder = ESM1bEncoder()

    wt_token_ids = {aa: encoder.tok_id(aa) for aa in 'ACDEFGHIKLMNPQRSTVWY'}

    rows, dn, llr_vals, slen = [], [], [], []
    try:
        from tqdm.auto import tqdm
        iterator = tqdm(df.itertuples(index=False), total=len(df), desc='Scoring')
    except ImportError:
        iterator = df.itertuples(index=False)

    for r in iterator:
        seq = wt_seqs[r.variant_id]
        pos1 = int(r.protein_pos) + 1  # 0-indexed → 1-indexed for HGVS
        seq_t, pos_t = truncate_around_mutation(seq, pos1, window=encoder.MAX_LEN)
        mut_str = f"{r.aa_ref}{pos_t}{r.aa_alt}"
        try:
            validate_mutation(seq_t, parse_mutation(mut_str))
            result = encode_variant(encoder, seq_t, mut_str)
        except Exception as e:
            print(f"  skip {r.variant_id}: {e}")
            continue
        rows.append((r.variant_id, r.gene, int(r.label)))
        dn.append(compute_delta_norm(result['wt_embedding'], result['mut_embedding']))
        # Brandes 2023 LLR: single WT pass; neg = deleterious.
        llr_vals.append(compute_llr(
            result['wt_logits'], result['mutation'],
            wt_token_ids[r.aa_ref], wt_token_ids[r.aa_alt],
        ))
        slen.append(len(seq))

    vid, gene, lbl = zip(*rows)
    scores_df = pd.DataFrame({
        'variant_id': vid, 'gene': gene, 'label': lbl,
        'delta_norm': dn, 'llr': llr_vals, 'seq_len': slen,
    })
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    np.savez(CACHE,
             variant_id=np.array(vid), gene=np.array(gene), label=np.array(lbl),
             delta_norm=np.array(dn), llr=np.array(llr_vals), seq_len=np.array(slen))
    print(f"Scored {len(scores_df)} variants, cached to {CACHE}")

scores_df.head()


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

# Bootstrap convention from ARCHITECTURE.md §5: n_resamples=10000, seed=42.
def bootstrap_auroc(y, s, n_boot=10000, seed=42):
    rng = np.random.default_rng(seed)
    aurocs = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        if len(np.unique(y[idx])) < 2:
            continue
        aurocs.append(roc_auc_score(y[idx], s[idx]))
    a = np.array(aurocs)
    return a.mean(), np.percentile(a, 2.5), np.percentile(a, 97.5)

y = scores_df['label'].values
scorers = {
    'Delta L2 norm': scores_df['delta_norm'].values,
    'LLR'          : scores_df['llr'].values,
}
summary = []
for name, s in scorers.items():
    m = ~np.isnan(s)
    a, lo, hi = bootstrap_auroc(y[m], s[m])
    summary.append({'score': name, 'AUROC': a, 'CI95_lo': lo, 'CI95_hi': hi, 'n': int(m.sum())})
summary_df = pd.DataFrame(summary).sort_values('AUROC', ascending=False)
print(summary_df.to_string(index=False))


In [ ]:
# Distributions: do P and B actually separate?
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (name, s) in zip(axes, scorers.items()):
    m = ~np.isnan(s)
    for lbl_val, color, lbl_name in [(0, '#4a8fb8', 'Benign'), (1, '#c44e52', 'Pathogenic')]:
        sub = s[m & (y == lbl_val)]
        sns.kdeplot(sub, ax=ax, color=color, fill=True, alpha=0.35, label=f'{lbl_name} (n={len(sub)})')
    ax.set_title(f'{name}  (AUROC = {roc_auc_score(y[m], s[m]):.3f})')
    ax.set_xlabel(name); ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ROC curves
fig, ax = plt.subplots(figsize=(7, 6))
for name, s in scorers.items():
    m = ~np.isnan(s)
    fpr, tpr, _ = roc_curve(y[m], s[m])
    auroc = roc_auc_score(y[m], s[m])
    ax.plot(fpr, tpr, lw=2, label=f'{name} — AUROC {auroc:.3f}')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Chance')
ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate')
ax.set_title('Live ESM-1b scoring of 500 ClinVar missense variants')
ax.legend(); plt.tight_layout(); plt.show()


In [ ]:
# Per-gene strip plot — restricted to genes with at least one P and one B
mixed = (
    scores_df.groupby('gene')['label'].nunique() == 2
)
mixed_genes = mixed[mixed].index.tolist()
print(f"Genes with both P and B in the validation set: {len(mixed_genes)}")
if mixed_genes:
    top = (
        scores_df[scores_df['gene'].isin(mixed_genes)]
        .groupby('gene').size().sort_values(ascending=False).head(15).index.tolist()
    )
    sub = scores_df[scores_df['gene'].isin(top)].copy()
    sub['label_name'] = sub['label'].map({0: 'Benign', 1: 'Pathogenic'})
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.stripplot(
        data=sub, x='gene', y='delta_norm', hue='label_name',
        palette={'Benign': '#4a8fb8', 'Pathogenic': '#c44e52'},
        dodge=True, jitter=0.25, ax=ax, size=6, alpha=0.85,
    )
    ax.set_title(f'Delta L2 norm per gene (top {len(top)} mixed-label genes)')
    ax.set_xlabel(''); ax.set_ylabel('Delta L2 norm')
    plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()
else:
    print("(no gene has both labels — strip plot skipped)")


In [ ]:
# Sanity: is delta_norm confounded by sequence length? (longer seq → larger pooled embedding shift?)
fig, ax = plt.subplots(figsize=(8, 5))
for lbl_val, color, name in [(0, '#4a8fb8', 'Benign'), (1, '#c44e52', 'Pathogenic')]:
    m = scores_df['label'] == lbl_val
    ax.scatter(scores_df.loc[m, 'seq_len'], scores_df.loc[m, 'delta_norm'],
               alpha=0.5, s=20, c=color, label=name)
ax.set_xlabel('Sequence length (aa)')
ax.set_ylabel('Delta L2 norm')
ax.set_title('Score vs sequence length — looking for confounding')
ax.legend(); plt.tight_layout(); plt.show()


**What you're seeing**: live ESM-1b scoring on 500 variants gives an AUROC for delta-norm
and LLR. Those numbers are the *honest answer* to "does ESM-1b separate ClinVar P/B at the
protein level?" — no caching, no cherry-picking, computed in front of you. Compare them
against the literature numbers on the slides.

**Why this matters for the handoff**: every box in this section — the score loop, the AUROC
bootstrap, the plots — is small enough to hand to an agent and large enough to be a real
unit of work. That's the next section.


---

## S4. Hand It Off

S2 + S3 produced a working prototype: `vep_utils.py` (the primitive), `validation_variants.csv`
+ `validation_proteins.fasta` (the data), and a notebook that scores 500 variants live and
visualizes the result. **That's enough for an agent to run with.**

What follows are three concrete handoffs you can paste into Claude Code (or your harness of
choice). Each one targets a different scaling axis: writing, sweeping, and serving. Pick one,
run it after the workshop, and have the prototype turned into something useful by tomorrow.


### Where to scale: the dispatcher skill

The workshop bundles a substrate router at `.claude/skills/dispatcher/`. Given a
workload spec (`n_items`, `requires_gpu`, `gpu_memory_gb`, `per_item_memory_gb`),
it picks where to run — laptop CPU, local GPU, Apple MPS, or a SLURM backend
you've registered — and emits the dispatch plan. The S4 prompts below reach for
it when the work needs to scale beyond what your machine can run directly.

**Three ways to invoke** (pick the one that fits your environment):

1. **Bash-direct** (always works):
   ```bash
   echo '{"n_items": 60, "requires_gpu": true, "gpu_memory_gb": 16, "per_item_memory_gb": 8}' \
       | python3 .claude/skills/dispatcher/scripts/route.py
   ```

2. **Project-local Claude Code discovery**: if your Claude Code client scans
   project-local `.claude/skills/`, the dispatcher is already available — paste
   any S4 prompt below and the Skill tool sees it.

3. **User-global Skill-tool discovery** (one-time setup): copy to your home dir
   so every Claude Code session on this machine sees it:
   ```bash
   mkdir -p ~/.claude/skills && cp -r .claude/skills/dispatcher ~/.claude/skills/
   ```

Backend manifests (for registering a SLURM cluster you have access to) live at
`.claude/skills/dispatcher/references/backends/<name>.json` — see
`_template.json` in that dir for the shape, and `SKILL.md` for full conventions.


### S4a. Writing — port to `expaper`

**Where it goes**: a fresh repo wired to your Overleaf project, with `vep_utils.py` and the
S3 scoring loop snapshotted as a reproducible experiment. Figures from S3 are committed to
`shared/figures/` and `\includegraphics`'d directly into the paper.

**What you do once, by hand**: create the Overleaf project (the API doesn't expose project
creation), copy the git URL, and run `expaper link-overleaf <url>` after init. The agent
handles everything else.

**What the agent does**: scaffold the repo, copy the prototype, register the tool, write a
first paragraph in `paper/main.tex` referencing the figures, push to Overleaf.


> 🤖 **Agent prompt — port to expaper**
>
> ```
> Use expaper to scaffold a research project around this workshop's variant-scoring prototype.
>
> Inputs:
> - This workshop's notebook (the S2/S3 prototype lives here)
> - The validation set at experiments/notebooks/data/workshop_set.tsv + manifest
> - Overleaf project git URL: <PASTE>
>
> Goal: scaffold the repo, register the prototype as a tool following expaper's
> conventions, generate the four S3 figures into the expected outputs path, write
> a one-paragraph Methods section referencing the figures via \includegraphics,
> push to Overleaf.
>
> CLAUDE.md and ARCHITECTURE.md spell out the conventions. Use them.
> Stop and ask if anything is ambiguous.
> ```


### S4b. Sweeping — 60 wandb runs from one prompt → paper figure

**The 0 → paper-figure arc, in one chain.** A Hydra+wandb sweep over **12 ESM-1b layers
× 5 score functions = 60 runs** fires live; the same agent prompt aggregates them into a
single AUROC-vs-layer figure ready to drop into a paper.

The biology: *which layer of ESM-1b carries the pathogenicity signal, and which readout
extracts it cleanly?* Early layers see local sequence; late layers see global structure;
the answer is empirical and the figure is the contribution.

**What you do once, by hand**: create a wandb project, export `WANDB_API_KEY`, tell the
agent your entity/project. **What the agent does, in one prompt**: write the sweep
config, launch 60 runs (visible live on the wandb dashboard), then fetch and plot.


> 🤖 **Agent prompt — sweep → fetch → plot**
>
> *"Sweep the variant-scoring prototype across ESM-1b layers × scorers to produce
> an AUROC-vs-layer figure for a Methods / Results section.
>
> Grid: layers {0, 3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 33} × the scorers in
> `manylatents.dogma.vep` (`delta_norm`, `cosine_dist`, `llr`, `lid`). The encoder
> exposes per-layer hidden states via `output_hidden_states=True`; index
> `hidden_states[layer]` for the layer-stratified embedding. The notebook's S2 /
> S3 cells show the scoring-loop pattern.
>
> Use the dispatcher (`.claude/skills/dispatcher/`) to pick where each run goes —
> local GPU / MPS for laptops, or a registered SLURM backend for cluster work.
> The dispatcher's workload-spec contract names the inputs (`n_items`,
> `requires_gpu`, `gpu_memory_gb`, `per_item_memory_gb`); fill in what one sweep
> run needs.
>
> Log to wandb (`<ENTITY>/<PROJECT>`) per the schema in `ARCHITECTURE.md` §5
> (`auroc`, `ci_lo`, `ci_hi`, `n_variants`, `layer`, `score`); bootstrap per the
> convention there. Aggregate into
> `experiments/analysis/figures/auroc_vs_layer.{pdf,png}` — one line per scorer,
> 95% CI bands, dashed line at AUROC = 0.5. Two-sentence interpretation pointing
> to where in ESM-1b the pathogenicity signal peaks. Append the wandb sweep URL
> to `experiments/EXPERIMENT_LOG.md` when the runs land."*


### S4c. Serving — hosted-model rewrite for the no-GPU path

**Where it goes**: a drop-in replacement encoder that calls a hosted inference API instead
of running ESM-1b locally. Same interface (`encode(seq) -> (embedding, logits)`), so
nothing else in the prototype changes.

**What you do once, by hand**: pick a hosted ESM provider and get an API key. Options
include the AWS HealthOmics ESM endpoint, a self-hosted vLLM/TGI deployment, or
HuggingFace Inference Endpoints (`facebook/esm1b_t33_650M_UR50S` is hostable).

**What the agent does**: write `HostedESM1bEncoder`, swap it into S2/S3, validate
embeddings match the local model on a small set, and document the latency/cost tradeoff.


> 🤖 **Agent prompt — hosted encoder**
>
> ```
> Add a hosted-inference path to the variant-scoring pipeline so workshop attendees
> without local GPUs can still run S2/S3.
>
> Inputs:
> - Hosted provider: <huggingface-inference | vllm | aws-healthomics | other>
> - Endpoint URL: <PASTE>
> - API key env var: HF_API_KEY (or equivalent)
>
> Goal: a hosted encoder that conforms to the encoder interface defined in
> manylatents.dogma.encoders. The S2/S3 cells should pick it up via a single
> config-level swap. Validate parity (cosine sim > 0.99 on 5 random variants,
> LLR abs-diff < 0.05). Report latency and per-1000-call cost vs local.
>
> The encoder interface and config conventions are in the repo. Match them.
> See `.claude/skills/dispatcher/SKILL.md` § "Hosted-encoder backends" for the
> env-var / retry / vocab-ordering conventions any hosted backend must follow.
> ```


**The pattern, named explicitly:**

1. **Prototype small** — one P + one B, eyeball the gap (S2).
2. **Validate at scale** — same prototype, ~500 variants, real AUROC (S3).
3. **Hand off** — the prototype is now the *spec* for the agent. Paper, sweep, serve
   — pick the axis and let the agent extend.

Every box in this notebook (a function, a CSV, a plot) was sized to be *small enough to
inspect* and *self-contained enough to ship*. That's the whole craft. Vocabulary, tooling
choices, harness internals — all in the slides. The notebook is the demonstration that the
workflow actually works.

Thanks for coming. Questions?
